## tl;dr

小维的灵活性主要来自高优先级接住直接互动、短程续聊、少量定向或环境参与，而不是随机主动发言。昵称是统一入口：先判断剩余内容属于能力还是社交，再进入对应链路。


## Context & Methods

- 数据范围：2026-07-23 至 2026-08-24 的目标群 JSONL 导出。
- 分析单位：去重消息、Bot 发言轮次和昵称呼唤消息。
- `next_bot` 是时间邻近指标，不等同于因果回复；`linked_reply` 只覆盖可解析引用关系。

### Key Assumptions

目标 Bot 使用 UIN `323537051` 识别；昵称分析使用“小维 / 小真寻 / 小真寻备用机”。


## Data


In [1]:
from pathlib import Path
import json
metrics_path = Path('analysis/target_bot_20260824/output/deep_behavior_metrics.json')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics['source'], ensure_ascii=False, indent=2))


{
  "rows_after_dedup": 16909,
  "observed_start": "2026-07-23T10:39:36+08:00",
  "observed_end": "2026-08-24T10:38:40+08:00",
  "bot_uin": "323537051",
  "bot_uid": "u_wa9kVWFd1RzFlh_blskvQQ",
  "bot_display_name": "小维"
}


## Results


In [2]:
print('称呼方式 | 样本 | 30秒内出现Bot发言 | 60秒内可解析引用回复 | 中位后续发言秒数')
print('-' * 88)
for label, row in metrics['addressing'].items():
    print(f"{label} | {row['messages']} | {row['next_bot_30s_rate']:.1%} | {row['linked_reply_60s_rate']:.1%} | {row['median_next_bot_seconds']}")


称呼方式 | 样本 | 30秒内出现Bot发言 | 60秒内可解析引用回复 | 中位后续发言秒数
----------------------------------------------------------------------------------------
明确@ | 452 | 87.2% | 59.5% | 13.0
昵称开头 | 171 | 86.5% | 24.0% | 10.0
昵称结尾 | 64 | 62.5% | 4.7% | 4.0
正文提及 | 583 | 57.5% | 21.6% | 13.0
昵称直接社交（排除明显能力请求） | 161 | 70.8% | 27.3% | 13.0


In [3]:
print('进入方式 | 轮次 | 占比 | P50/P90秒 | 2分钟明确回访')
print('-' * 76)
for row in metrics['trigger_lanes']:
    p50 = row['median_latency_seconds']
    p90 = row['p90_latency_seconds']
    print(f"{row['lane']} | {row['turns']} | {row['share']:.1%} | {p50}/{p90} | {row['explicit_followup_2m_rate']:.1%}")


进入方式 | 轮次 | 占比 | P50/P90秒 | 2分钟明确回访
----------------------------------------------------------------------------
被点名/被回复 | 447 | 27.1% | 5.0/17.0 | 38.9%
自主定向回复 | 204 | 12.4% | 5.5/18.7 | 27.9%
连续对话 | 797 | 48.4% | 3.0/10.0 | 18.3%
环境话题加入 | 189 | 11.5% | 4.0/14.0 | 11.1%
主动开场 | 2 | 0.1% | None/None | 0.0%
延迟上下文 | 8 | 0.5% | 151.5/256.4 | 25.0%


In [4]:
shape = metrics['turn_shape']
style = metrics['style']
targets = metrics['recipient_concentration']
print(f"平均每轮消息数: {shape['mean_messages_per_turn']}")
print(f"多消息轮次: {shape['multi_message_turn_rate']:.1%}")
print(f"轮次文本长度 P50/P90: {shape['median_turn_chars']}/{shape['p90_turn_chars']:.1f} 字")
print(f"多轮对话链: {shape['multi_turn_dialogue_chains']}，链长 P50/P90: {shape['median_chain_turns']}/{shape['p90_chain_turns']:.1f}")
print(f"文本唯一率: {style['unique_text_ratio']:.1%}，语气词覆盖: {style['soft_particle_rate']:.1%}")
print(f"可解析定向对象: {targets['unique_reply_recipients']}，Top1占比: {targets['top_1_share']:.1%}")


平均每轮消息数: 1.42
多消息轮次: 32.1%
轮次文本长度 P50/P90: 14.0/53.4 字
多轮对话链: 344，链长 P50/P90: 3.0/5.7
文本唯一率: 83.5%，语气词覆盖: 54.0%
可解析定向对象: 95，Top1占比: 9.4%


## Takeaways

1. `@`、回复和高置信昵称呼唤应走确定性的直接互动通道；模型负责理解和表达，不负责否认“对方在叫我”。
2. 昵称识别后必须再次判断剩余文本归属：外部能力优先，普通表达进入闲聊。
3. 直接互动应打开短期对话租约，后续 3–6 轮优先延续，而不是每条消息从零判断是否参与。
4. 环境参与保持少量、定向、受频控；冷启动主动开场维持极低频。
5. 输出默认 1–2 条短消息，媒体是表达能力而不是核心参与机制。
